# Data preparation

In [2]:
# import libraries
import pandas as pd
import numpy as np


In [3]:
# loading the  fear and greed dataset
fg = pd.read_csv("data\\fear_greed_index.csv", parse_dates=["date"])
fg.head()

,timestamp,value,classification,date
0,1517463000,30,Fear,2018-02-01
1,1517549400,15,Extreme Fear,2018-02-02
2,1517635800,40,Fear,2018-02-03
3,1517722200,24,Extreme Fear,2018-02-04
4,1517808600,11,Extreme Fear,2018-02-05


In [4]:
print(f"Shape: {fg.shape}")         
print(fg.dtypes)                     
print(fg.isnull().sum())             
print(fg.duplicated().sum())         

Shape: (2644, 4)
timestamp                  int64
value                      int64
classification            object
date              datetime64[ns]
dtype: object
timestamp         0
value             0
classification    0
date              0
dtype: int64
0


In [5]:
# loading the trader dataset
trader = pd.read_csv("data\\historical_data.csv")
trader.head(2)

,Account,Coin,Execution Price,Size Tokens,Size USD,Side,Timestamp IST,Start Position,Direction,Closed PnL,Transaction Hash,Order ID,Crossed,Fee,Trade ID,Timestamp
0,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9769,986.87,7872.16,BUY,02-12-2024 22:50,0.000000,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.345404,8.950000e+14,1.730000e+12
1,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9800,16.00,127.68,BUY,02-12-2024 22:50,986.524596,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.005600,4.430000e+14,1.730000e+12


In [6]:
print(f"Shape: {trader.shape}")
print(trader.dtypes)
print(trader.isnull().sum())
print(trader.duplicated().sum())

Shape: (211224, 16)
Account              object
Coin                 object
Execution Price     float64
Size Tokens         float64
Size USD            float64
Side                 object
Timestamp IST        object
Start Position      float64
Direction            object
Closed PnL          float64
Transaction Hash     object
Order ID              int64
Crossed                bool
Fee                 float64
Trade ID            float64
Timestamp           float64
dtype: object
Account             0
Coin                0
Execution Price     0
Size Tokens         0
Size USD            0
Side                0
Timestamp IST       0
Start Position      0
Direction           0
Closed PnL          0
Transaction Hash    0
Order ID            0
Crossed             0
Fee                 0
Trade ID            0
Timestamp           0
dtype: int64
0


In [7]:
# Parse the human-readable IST timestamp
# format="%d-%m-%Y %H:%M" is explicit — never let pandas guess
# because 02-12-2024 could be read as Feb 12 (wrong) or Dec 2 (correct)
trader["Timestamp IST"] = pd.to_datetime(
    trader["Timestamp IST"],
    format="%d-%m-%Y %H:%M"
)

print(f"Timestamp IST dtype: {trader['Timestamp IST'].dtype}")
print(trader["Timestamp IST"].head(3))

Timestamp IST dtype: datetime64[ns]
0   2024-12-02 22:50:00
1   2024-12-02 22:50:00
2   2024-12-02 22:50:00
Name: Timestamp IST, dtype: datetime64[ns]


In [8]:
# Before trusting or dropping the Unix Timestamp column
# check how many unique values it has
print(f"Unique Timestamp values: {trader['Timestamp'].nunique()}")
print(trader["Timestamp"].value_counts())

Unique Timestamp values: 7
Timestamp
1.740000e+12    133871
1.730000e+12     35241
1.750000e+12     26961
1.720000e+12      7141
1.710000e+12      6962
1.700000e+12      1045
1.680000e+12         3
Name: count, dtype: int64


In [9]:
# Convert Unix timestamp (milliseconds) to IST for comparison
trader["ts_unix_check"] = pd.to_datetime(
    trader["Timestamp"], unit="ms", utc=True
).dt.tz_convert("Asia/Kolkata").dt.tz_localize(None).dt.floor("min")

# Compare dates from both columns
trader["date_from_ist"]  = trader["Timestamp IST"].dt.date
trader["date_from_unix"] = trader["ts_unix_check"].dt.date

date_mismatch = trader["date_from_ist"] != trader["date_from_unix"]

print(f"Total rows:        {len(trader)}")
print(f"Date mismatches:   {date_mismatch.sum()}")
print(f"Mismatch %:        {date_mismatch.mean()*100:.1f}%")

print("\nDate distribution from Timestamp IST:")
print(trader["date_from_ist"].value_counts().sort_index())

print("\nDate distribution from Unix Timestamp:")
print(trader["date_from_unix"].value_counts().sort_index())

Total rows:        211224
Date mismatches:   209073
Mismatch %:        99.0%

Date distribution from Timestamp IST:
date_from_ist
2023-05-01       3
2023-12-05       9
2023-12-14      11
2023-12-15       2
2023-12-16       3
              ... 
2025-04-27     337
2025-04-28    1379
2025-04-29    2243
2025-04-30    1113
2025-05-01    1230
Name: count, Length: 480, dtype: int64

Date distribution from Unix Timestamp:
date_from_unix
2023-03-28         3
2023-11-15      1045
2024-03-09      6962
2024-07-03      7141
2024-10-27     35241
2025-02-20    133871
2025-06-15     26961
Name: count, dtype: int64


### Summarizing  what I found
`FINDING:` Unix Timestamp column is corrupted.

`Root cause:` CSV was exported via Excel/Google Sheets which
auto-formatted the 13-digit millisecond Unix timestamp
(e.g. 1,733,081,447,382) into scientific notation
(1.730000e+12), rounding away the last 9 digits and
collapsing 211,224 unique timestamps into only 7 values.

`Example:`
  Real timestamp:    1,733,081,447,382  (unique per trade)
  After Excel:       1,730,000,000,000  (all Nov-Dec trades identical)

This is irreversible the original precision cannot be recovered.

`Decision:` Drop the Unix Timestamp column entirely.
          Use Timestamp IST as the sole time reference.
          Timestamp IST shows a clean distribution across
          Oct 2024 - May 2025 with realistic daily trade counts.


In [10]:
# Drop the corrupted Unix Timestamp and all investigation columns
cols_to_drop = [c for c in ["Timestamp", "ts_unix_check",
                             "date_from_ist", "date_from_unix"]
                if c in trader.columns]

trader.drop(columns=cols_to_drop, inplace=True)

# Extract clean date column for merging with Fear/Greed data
trader["date"] = pd.to_datetime(trader["Timestamp IST"].dt.date)

print(f"Columns remaining: {len(trader.columns)}")
print(trader.columns.tolist())
print(f"\nDate range: {trader['date'].min()} to {trader['date'].max()}")
print(f"Unique trade days: {trader['date'].nunique()}")
print("\nTrades per day (sample):")
print(trader["date"].value_counts().sort_index().tail(10))

Columns remaining: 16
['Account', 'Coin', 'Execution Price', 'Size Tokens', 'Size USD', 'Side', 'Timestamp IST', 'Start Position', 'Direction', 'Closed PnL', 'Transaction Hash', 'Order ID', 'Crossed', 'Fee', 'Trade ID', 'date']

Date range: 2023-05-01 00:00:00 to 2025-05-01 00:00:00
Unique trade days: 480

Trades per day (sample):
date
2025-04-22    2998
2025-04-23    6159
2025-04-24    2232
2025-04-25    1653
2025-04-26    1131
2025-04-27     337
2025-04-28    1379
2025-04-29    2243
2025-04-30    1113
2025-05-01    1230
Name: count, dtype: int64


In [11]:
# Confirm date extraction looks correct
# Should see realistic daily trade counts, no outlier dates
# Data starts May 2023, not 2024 as originally assumed

assert trader["date"].isnull().sum() == 0, \
    "Nulls found in date column"

assert trader["date"].min() >= pd.Timestamp("2023-01-01"), \
    "Dates unrealistically old — pre 2023"

assert trader["date"].max() < pd.Timestamp("2026-01-01"), \
    "Future dates found"

assert trader["date"].nunique() == 480, \
    f"Expected 480 unique dates, got {trader['date'].nunique()}"

print("Sanity checks passed:")
print(f"  No null dates")
print(f"  Min date: {trader['date'].min().date()}")
print(f"  Max date: {trader['date'].max().date()}")
print(f"  Unique trading days: {trader['date'].nunique()}")


Sanity checks passed:
  No null dates
  Min date: 2023-05-01
  Max date: 2025-05-01
  Unique trading days: 480


In [12]:
merged = pd.merge(
    trader,
    fg[["date", "value", "classification"]],
    on="date",
    how="left"
)
print(f"Trader rows before merge:  {len(trader)}")
print(f"Merged rows after merge:   {len(merged)}")
print(f"Columns after merge:       {merged.columns.tolist()}")
print(f"Merged shape: {merged.shape}")
print(merged["classification"].value_counts())
print(merged["classification"].isnull().sum())  # trades with no sentiment match
merged.head(2)

Trader rows before merge:  211224
Merged rows after merge:   211224
Columns after merge:       ['Account', 'Coin', 'Execution Price', 'Size Tokens', 'Size USD', 'Side', 'Timestamp IST', 'Start Position', 'Direction', 'Closed PnL', 'Transaction Hash', 'Order ID', 'Crossed', 'Fee', 'Trade ID', 'date', 'value', 'classification']
Merged shape: (211224, 18)
classification
Fear             61837
Greed            50303
Extreme Greed    39992
Neutral          37686
Extreme Fear     21400
Name: count, dtype: int64
6


,Account,Coin,Execution Price,Size Tokens,Size USD,Side,Timestamp IST,Start Position,Direction,Closed PnL,Transaction Hash,Order ID,Crossed,Fee,Trade ID,date,value,classification
0,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9769,986.87,7872.16,BUY,2024-12-02 22:50:00,0.000000,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.345404,8.950000e+14,2024-12-02,80.0,Extreme Greed
1,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9800,16.00,127.68,BUY,2024-12-02 22:50:00,986.524596,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.005600,4.430000e+14,2024-12-02,80.0,Extreme Greed


In [13]:
print("=== Sentiment coverage ===")
print(f"Rows with sentiment label: {merged['classification'].notnull().sum()}")
print(f"Rows without label (NaN):  {merged['classification'].isnull().sum()}")
print(f"Coverage %:                {merged['classification'].notnull().mean()*100:.2f}%")

print("\nSentiment distribution:")
dist = merged["classification"].value_counts()
dist_pct = merged["classification"].value_counts(normalize=True).mul(100).round(1)

print(pd.DataFrame({"count": dist, "percent": dist_pct}))

=== Sentiment coverage ===
Rows with sentiment label: 211218
Rows without label (NaN):  6
Coverage %:                100.00%

Sentiment distribution:
                count  percent
classification                
Fear            61837     29.3
Greed           50303     23.8
Extreme Greed   39992     18.9
Neutral         37686     17.8
Extreme Fear    21400     10.1


In [14]:
# Check which dates have missing sentiment labels
unmatched = merged[merged["classification"].isnull()]

if len(unmatched) == 0:
    print("All rows matched — 100% coverage")
else:
    print(f"{len(unmatched)} unmatched rows")
    print("\nDates with no sentiment match:")
    print(unmatched["date"].value_counts())

6 unmatched rows

Dates with no sentiment match:
date
2024-10-26    6
Name: count, dtype: int64


#### jump in the fear/greed data
`FINDING:` 6 trades on 2024-10-26 had no matching Fear/Greed entry.
Investigation showed Oct 26 is simply absent from the FG dataset
(jumps from Oct 25 to Oct 27 a known occasional gap in the index).

`Fix:` Forward-filled from Oct 25 sentiment (Greed, value=72).
This is standard practice for single-day gaps in daily indices.
Oct 27 reading was also Greed (74) confirming the fill is accurate.

Rows affected: 6 (0.003% of total)


In [15]:
# Modern syntax — replaces fillna(method="ffill")
merged["classification"] = merged["classification"].ffill()
merged["value"]          = merged["value"].ffill()

# Verify
print(f"Unmatched rows after fill: {merged['classification'].isnull().sum()}")
print(merged[merged["date"] == "2024-10-26"][
    ["date", "classification", "value"]
].drop_duplicates())

Unmatched rows after fill: 0
          date classification  value
727 2024-10-26          Greed   72.0


In [16]:
print("=== Final merged dataset ===")
print(f"Shape:            {merged.shape}")
print(f"Columns:          {merged.columns.tolist()}")
print(f"Date range:       {merged['date'].min().date()} to {merged['date'].max().date()}")
print(f"Unique accounts:  {merged['Account'].nunique()}")
print(f"Unique coins:     {merged['Coin'].nunique()}")
print(f"Sentiment cover:  {merged['classification'].notnull().mean()*100:.1f}%")
print(f"\nFirst 3 rows:")
print(merged[["Account", "date", "Closed PnL", "classification", "value"]].head(3))

=== Final merged dataset ===
Shape:            (211224, 18)
Columns:          ['Account', 'Coin', 'Execution Price', 'Size Tokens', 'Size USD', 'Side', 'Timestamp IST', 'Start Position', 'Direction', 'Closed PnL', 'Transaction Hash', 'Order ID', 'Crossed', 'Fee', 'Trade ID', 'date', 'value', 'classification']
Date range:       2023-05-01 to 2025-05-01
Unique accounts:  32
Unique coins:     246
Sentiment cover:  100.0%

First 3 rows:
                                      Account       date  Closed PnL  \
0  0xae5eacaf9c6b9111fd53034a602c192a04e082ed 2024-12-02         0.0   
1  0xae5eacaf9c6b9111fd53034a602c192a04e082ed 2024-12-02         0.0   
2  0xae5eacaf9c6b9111fd53034a602c192a04e082ed 2024-12-02         0.0   

  classification  value  
0  Extreme Greed   80.0  
1  Extreme Greed   80.0  
2  Extreme Greed   80.0  


In [17]:
print("=== Sentiment distribution across all trades ===")
dist = merged["classification"].value_counts()
dist_pct = merged["classification"].value_counts(normalize=True).mul(100).round(1)

print(pd.DataFrame({"trade count": dist, "% of trades": dist_pct}))

print("\n======= Simplified Fear vs Greed split ========")
# Mapping all 5 categories into 3 buckets for cleaner analysis
merged["sentiment"] = merged["classification"].map({
    "Extreme Fear" : "Fear",
    "Fear"         : "Fear",
    "Neutral"      : "Neutral",
    "Greed"        : "Greed",
    "Extreme Greed": "Greed"
})

simple_dist = merged["sentiment"].value_counts()
simple_pct  = merged["sentiment"].value_counts(normalize=True).mul(100).round(1)
print(pd.DataFrame({"trade count": simple_dist, "% of trades": simple_pct}))

=== Sentiment distribution across all trades ===
                trade count  % of trades
classification                          
Fear                  61837         29.3
Greed                 50309         23.8
Extreme Greed         39992         18.9
Neutral               37686         17.8
Extreme Fear          21400         10.1

======= Simplified Fear vs Greed split ========
           trade count  % of trades
sentiment                          
Greed            90301         42.8
Fear             83237         39.4
Neutral          37686         17.8


#### FINDING 1 — Sentiment distribution across 211,224 trades

Contrary to expectation, Fear days (29.3%) generate more individual 
trades than any other single category, including Greed (23.8%).

Combined buckets: Greed 42.8% | Fear 39.4% | Neutral 17.8%


Key question this raises: are Fear-day trades actually profitable,
or are traders overtrading during panic?

# Analysis 

In [18]:
# Step 1 — create helper columns on the merged dataset
merged["is_win"]  = merged["Closed PnL"] > 0
merged["is_long"] = merged["Side"].str.upper() == "BUY"

# Step 2 — aggregate to daily level per account
daily = merged.groupby(
    ["Account", "date", "classification", "sentiment", "value"]
).agg(
    daily_pnl      = ("Closed PnL",  "sum"),
    trade_count    = ("Closed PnL",  "count"),
    win_rate       = ("is_win",       "mean"),
    avg_size_usd   = ("Size USD",     "mean"),
    total_size_usd = ("Size USD",     "sum"),
    long_ratio     = ("is_long",      "mean"),
    total_fees     = ("Fee",          "sum"),
).reset_index()

print(f"Daily aggregated shape: {daily.shape}")
print(f"\nSample:")
print(daily[["Account", "date", "sentiment", "daily_pnl", 
             "trade_count", "win_rate", "long_ratio"]].head(5))
print(f"\nDaily metrics summary:")
print(daily[["daily_pnl", "trade_count", 
             "win_rate", "avg_size_usd"]].describe().round(2))

Daily aggregated shape: (2341, 12)

Sample:
                                      Account       date sentiment  daily_pnl  \
0  0x083384f897ee0f19899168e3b1bec365f52a9012 2024-11-11     Greed        0.0   
1  0x083384f897ee0f19899168e3b1bec365f52a9012 2024-11-17     Greed        0.0   
2  0x083384f897ee0f19899168e3b1bec365f52a9012 2024-11-18     Greed        0.0   
3  0x083384f897ee0f19899168e3b1bec365f52a9012 2024-11-22     Greed   -21227.0   
4  0x083384f897ee0f19899168e3b1bec365f52a9012 2024-11-26     Greed     1603.1   

   trade_count  win_rate  long_ratio  
0          177  0.000000    0.000000  
1           68  0.000000    0.000000  
2           40  0.000000    0.000000  
3           12  0.000000    1.000000  
4           27  0.444444    0.444444  

Daily metrics summary:
       daily_pnl  trade_count  win_rate  avg_size_usd
count    2341.00      2341.00   2341.00       2341.00
mean     4398.53        90.23      0.36       6989.52
std     28415.94       214.61      0.34      2153

In [19]:
# How many daily rows have win_rate = 0 because ALL trades were zero PnL
all_zero_days = daily[daily["win_rate"] == 0]
print(f"Account-days with 0% win rate: {len(all_zero_days)}")
print(f"As % of total:                 {len(all_zero_days)/len(daily)*100:.1f}%")

# Are these days with genuinely zero PnL or open positions?
print(f"\nPnL on these zero win-rate days:")
print(all_zero_days["daily_pnl"].describe().round(2))
# Find the extreme days
print("Top 5 best days:")
print(daily.nlargest(5, "daily_pnl")[
    ["Account", "date", "sentiment", "daily_pnl", "trade_count"]
])

print("\nTop 5 worst days:")
print(daily.nsmallest(5, "daily_pnl")[
    ["Account", "date", "sentiment", "daily_pnl", "trade_count"]
])

Account-days with 0% win rate: 748
As % of total:                 32.0%

PnL on these zero win-rate days:
count       748.00
mean      -1128.05
std        9571.88
min     -175611.00
25%           0.00
50%           0.00
75%           0.00
max           0.00
Name: daily_pnl, dtype: float64
Top 5 best days:
                                         Account       date sentiment  \
21    0x083384f897ee0f19899168e3b1bec365f52a9012 2025-03-03      Fear   
1998  0xb1231a4a2dd02f2276fa3c5e2a2f3436e6bfed23 2024-12-12     Greed   
12    0x083384f897ee0f19899168e3b1bec365f52a9012 2025-02-04     Greed   
1094  0x72743ae2822edd658c0c50608fd7c5c501b2afbd 2024-12-22     Greed   
23    0x083384f897ee0f19899168e3b1bec365f52a9012 2025-04-12      Fear   

          daily_pnl  trade_count  
21    533974.662903          641  
1998  449328.107544          314  
12    375620.270243          531  
1094  327545.999992          104  
23    307855.803410           31  

Top 5 worst days:
                         

In [23]:
# Total PnL per account across all time
account_summary = merged.groupby("Account").agg(
    total_pnl    = ("Closed PnL", "sum"),
    total_trades = ("Closed PnL", "count"),
    trading_days = ("date",       "nunique"),
    avg_pnl_per_day = ("Closed PnL", lambda x: 
                        x.sum() / merged.loc[x.index, "date"].nunique())
).reset_index()

# Sort by total PnL
account_summary = account_summary.sort_values(
    "total_pnl", ascending=False
).reset_index(drop=True)

print("=== All 32 accounts ranked by total PnL ===")
print(account_summary[[
    "Account", "total_pnl", "total_trades", "trading_days"
]].to_string())

=== All 32 accounts ranked by total PnL ===
                                       Account     total_pnl  total_trades  trading_days
0   0xb1231a4a2dd02f2276fa3c5e2a2f3436e6bfed23  2.143383e+06         14733           256
1   0x083384f897ee0f19899168e3b1bec365f52a9012  1.600230e+06          3818            24
2   0xbaaaf6571ab7d571043ff1e313a9609a10637864  9.401638e+05         21192            28
3   0x513b8629fe877bb581bf244e326a047b249c4ff1  8.404226e+05         12236            39
4   0xbee1707d6b44d4d52bfe19e41f8a828645437aab  8.360806e+05         40184           131
5   0x4acb90e786d897ecffb614dc822eb231b4ffb9f4  6.777471e+05          4356            58
6   0x72743ae2822edd658c0c50608fd7c5c501b2afbd  4.293556e+05          1590            19
7   0x430f09841d65beb3f27765503d0f850b8bce7713  4.165419e+05          1237            28
8   0x72c6a4624e1dffa724e6d00d64ceae698af892a0  4.030115e+05          1430            66
9   0x75f7eeb85dc639d5e99c78f95393aa9a5f1170d4  3.790954e+05      

In [24]:
# What % of all trades does each account represent?
account_summary["trade_share_pct"] = (
    account_summary["total_trades"] / len(merged) * 100
).round(1)

print("\n=== Trade volume concentration ===")
print(account_summary[[
    "Account", "total_trades", "trade_share_pct", "total_pnl"
]].head(10).to_string())

# How much do top 5 accounts represent?
top5_trades = account_summary["total_trades"].head(5).sum()
top5_pnl    = account_summary["total_pnl"].head(5).sum()

print(f"\nTop 5 accounts:")
print(f"  Trade share: {top5_trades/len(merged)*100:.1f}% of all trades")
print(f"  PnL share:   {top5_pnl/merged['Closed PnL'].sum()*100:.1f}% of total PnL")


=== Trade volume concentration ===
                                      Account  total_trades  trade_share_pct     total_pnl
0  0xb1231a4a2dd02f2276fa3c5e2a2f3436e6bfed23         14733              7.0  2.143383e+06
1  0x083384f897ee0f19899168e3b1bec365f52a9012          3818              1.8  1.600230e+06
2  0xbaaaf6571ab7d571043ff1e313a9609a10637864         21192             10.0  9.401638e+05
3  0x513b8629fe877bb581bf244e326a047b249c4ff1         12236              5.8  8.404226e+05
4  0xbee1707d6b44d4d52bfe19e41f8a828645437aab         40184             19.0  8.360806e+05
5  0x4acb90e786d897ecffb614dc822eb231b4ffb9f4          4356              2.1  6.777471e+05
6  0x72743ae2822edd658c0c50608fd7c5c501b2afbd          1590              0.8  4.293556e+05
7  0x430f09841d65beb3f27765503d0f850b8bce7713          1237              0.6  4.165419e+05
8  0x72c6a4624e1dffa724e6d00d64ceae698af892a0          1430              0.7  4.030115e+05
9  0x75f7eeb85dc639d5e99c78f95393aa9a5f1170d4         

In [20]:
# Recalculate win rate on CLOSED trades only (non-zero PnL)
merged["is_closed"] = merged["Closed PnL"] != 0
merged["is_win_closed"] = (merged["Closed PnL"] > 0) & merged["is_closed"]

daily_v2 = merged.groupby(
    ["Account", "date", "classification", "sentiment", "value"]
).agg(
    daily_pnl         = ("Closed PnL",     "sum"),
    trade_count       = ("Closed PnL",     "count"),
    closed_trade_count= ("is_closed",      "sum"),
    win_rate          = ("is_win_closed",  lambda x: 
                         x.sum() / merged.loc[x.index, "is_closed"].sum() 
                         if merged.loc[x.index, "is_closed"].sum() > 0 
                         else np.nan),
    avg_size_usd      = ("Size USD",       "mean"),
    total_size_usd    = ("Size USD",       "sum"),
    long_ratio        = ("is_long",        "mean"),
    total_fees        = ("Fee",            "sum"),
).reset_index()

print(f"Shape: {daily_v2.shape}")
print(f"\nWin rate comparison:")
print(f"Old win rate (includes zeros) - mean: {daily['win_rate'].mean():.3f}")
print(f"New win rate (closed only)    - mean: {daily_v2['win_rate'].mean():.3f}")
print(f"NaN win rate days (all open): {daily_v2['win_rate'].isna().sum()}")


Shape: (2341, 13)

Win rate comparison:
Old win rate (includes zeros) - mean: 0.360
New win rate (closed only)    - mean: 0.848
NaN win rate days (all open): 648


In [ ]:
# Profile the 0x0833 account specifically
whale = merged[merged["Account"] == "0x083384f897ee0f19899168e3b1bec365f52a9012"]

print("=== Account 0x083384f897ee0f19899168e3b1bec365f52a9012 profile ===")
print(f"Total trades:      {len(whale)}")
print(f"% of all trades:   {len(whale)/len(merged)*100:.1f}%")
print(f"Total PnL:         {whale['Closed PnL'].sum():,.2f}")
print(f"Date range:        {whale['date'].min().date()} to {whale['date'].max().date()}")
print(f"Avg trades/day:    {len(whale)/whale['date'].nunique():.0f}")
print(f"\nPnL by sentiment:")
print(whale.groupby("sentiment")["Closed PnL"].agg(["sum","mean","count"]).round(2))


=== Account 0x0833 profile ===
Total trades:      3818
% of all trades:   1.8%
Total PnL:         1,600,229.82
Date range:        2024-11-11 to 2025-04-12
Avg trades/day:    159

PnL by sentiment:
                  sum    mean  count
sentiment                           
Fear       1238142.77  659.29   1878
Greed       236436.93  155.65   1519
Neutral     125650.12  298.46    421


FINDING 1 — Dataset is dominated by one high-volume account

Account 0x0833 appears in 2 of the top 3 best trading days,
including the all-time best day (+$533k on a Fear day).
This account likely represents a high-frequency or algorithmic
trader and will be treated as a separate segment in Part B.

32% of account-days (748/2341) have win_rate=0 due to open
positions with Closed PnL=0. Win rate metric recalculated on
closed trades only to avoid this distortion.